# Analyse du succès commercial des films Bollywood

Ce notebook étudie le succès commercial des films Bollywood à partir de quatre variables principales : `Votes`, `Budget(INR)`, `Number of Screens` et `Rating(10)`. L'objectif est double :

1. comprendre le profil quantitatif des films ayant réussi ;
2. entraîner des modèles capables d'estimer la classe de succès probable d'un nouveau film.

La variable cible est construite à partir de `Revenue(INR)` en trois classes : `Flop`, `Average`, `Hit`.

In [4]:
import warnings, math, os, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = "bollywood_merged_clean.csv"


## 1. Chargement des données

In [ ]:
df = pd.read_csv("../Bollywood_Movies_data//bollywood_merged_clean.csv")
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'bollywood_merged_clean.csv'

## 2. Vérification rapide

In [ ]:
print(df.info())
print("\nValeurs manquantes :")
print(df.isna().sum())
print("\nDescription statistique :")
df[['Number of Screens','Revenue(INR)','Budget(INR)','Rating(10)','Votes']].describe()

## 3. Création de la cible `Success_Class`

Nous transformons `Revenue(INR)` en trois classes à l'aide des terciles afin d'obtenir un problème de classification.

In [ ]:
q1, q2 = df['Revenue(INR)'].quantile([1/3, 2/3])

def revenue_to_class(x):
    if x <= q1:
        return 'Flop'
    elif x <= q2:
        return 'Average'
    return 'Hit'

df['Success_Class'] = df['Revenue(INR)'].apply(revenue_to_class)
df['Success_Class'].value_counts()

## 4. Sélection des variables principales

In [ ]:
features = ['Votes', 'Budget(INR)', 'Number of Screens', 'Rating(10)']
target = 'Success_Class'

work = df[features + [target]].copy()

# transformations logarithmiques pour limiter l'asymétrie
work['Votes_log'] = np.log1p(work['Votes'])
work['Budget_log'] = np.log1p(work['Budget(INR)'])
work['Screens_log'] = np.log1p(work['Number of Screens'])

model_features = ['Votes_log', 'Budget_log', 'Screens_log', 'Rating(10)']
work.head()

## 5. Analyse exploratoire

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), ['Votes', 'Budget(INR)', 'Number of Screens', 'Rating(10)']):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'Distribution de {col}')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), ['Votes', 'Budget(INR)', 'Number of Screens', 'Rating(10)']):
    sns.boxplot(data=df.assign(Success_Class=work[target]), x='Success_Class', y=col, ax=ax)
    ax.set_title(f'{col} selon la classe de succès')
plt.tight_layout()
plt.show()

In [ ]:
corr_df = df[['Votes','Budget(INR)','Number of Screens','Rating(10)','Revenue(INR)']].copy()
plt.figure(figsize=(8, 5))
sns.heatmap(corr_df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de corrélation")
plt.show()

## 6. Préparation train/test

In [ ]:
X = work[model_features].copy()
y = work[target].copy()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Classes:", list(label_encoder.classes_))

## 7. Modèle 1 — Gaussian Naive Bayes

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

gnb_pred = gnb.predict(X_test)
gnb_proba = gnb.predict_proba(X_test)

gnb_results = {
    'accuracy': accuracy_score(y_test, gnb_pred),
    'precision_macro': precision_score(y_test, gnb_pred, average='macro'),
    'recall_macro': recall_score(y_test, gnb_pred, average='macro'),
    'f1_macro': f1_score(y_test, gnb_pred, average='macro')
}
gnb_results

In [ ]:
print(classification_report(y_test, gnb_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, gnb_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Matrice de confusion - GaussianNB')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

## 8. Modèle 2 — FT-Transformer simplifié

Pour rester dans un notebook autonome, on utilise une version légère inspirée des Transformers pour données tabulaires. Les quatre variables numériques sont projetées, combinées, puis traitées par un bloc Transformer.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabularDataset(X_train_scaled, y_train)
test_ds = TabularDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

class SimpleFTTransformer(nn.Module):
    def __init__(self, n_features, n_classes, d_model=32, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.feature_embed = nn.Linear(1, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=64,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )
    def forward(self, x):
        # x: [batch, n_features]
        x = x.unsqueeze(-1)              # [batch, n_features, 1]
        x = self.feature_embed(x)        # [batch, n_features, d_model]
        x = self.transformer(x)          # [batch, n_features, d_model]
        x = x.mean(dim=1)                # pooling
        x = self.norm(x)
        return self.classifier(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleFTTransformer(n_features=X_train.shape[1], n_classes=len(label_encoder.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_targets), np.array(all_preds), np.array(all_probs)

history = []
epochs = 35

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)

    train_loss = running_loss / len(train_loader.dataset)
    yt, yp, _ = evaluate_model(model, test_loader)
    f1 = f1_score(yt, yp, average='macro')
    history.append((epoch + 1, train_loss, f1))

pd.DataFrame(history, columns=['epoch', 'train_loss', 'test_f1']).tail()

In [ ]:
hist_df = pd.DataFrame(history, columns=['epoch', 'train_loss', 'test_f1'])
plt.figure(figsize=(7,4))
plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train loss')
plt.plot(hist_df['epoch'], hist_df['test_f1'], label='Test F1 macro')
plt.legend()
plt.title('Entraînement du FT-Transformer simplifié')
plt.show()

In [ ]:
ft_y_true, ft_pred, ft_probs = evaluate_model(model, test_loader)

ft_results = {
    'accuracy': accuracy_score(ft_y_true, ft_pred),
    'precision_macro': precision_score(ft_y_true, ft_pred, average='macro'),
    'recall_macro': recall_score(ft_y_true, ft_pred, average='macro'),
    'f1_macro': f1_score(ft_y_true, ft_pred, average='macro')
}
ft_results

In [ ]:
print(classification_report(ft_y_true, ft_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(ft_y_true, ft_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Matrice de confusion - FT-Transformer simplifié')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

## 9. Comparaison des modèles

In [ ]:
comparison = pd.DataFrame([gnb_results, ft_results], index=['GaussianNB', 'FT-Transformer'])
comparison.sort_values('f1_macro', ascending=False)

## 10. Fonction de prédiction pour un nouveau film

In [ ]:
def predict_new_film(votes, budget, screens, rating):
    row = pd.DataFrame([{
        'Votes_log': np.log1p(votes),
        'Budget_log': np.log1p(budget),
        'Screens_log': np.log1p(screens),
        'Rating(10)': rating
    }])

    # Naive Bayes
    nb_class = label_encoder.inverse_transform(gnb.predict(row))[0]
    nb_probs = gnb.predict_proba(row)[0]

    # FT-Transformer
    row_scaled = scaler.transform(row)
    tensor = torch.tensor(row_scaled, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
        pred = np.argmax(probs)
    ft_class = label_encoder.inverse_transform([pred])[0]

    return {
        'GaussianNB_class': nb_class,
        'GaussianNB_probs': dict(zip(label_encoder.classes_, nb_probs)),
        'FTTransformer_class': ft_class,
        'FTTransformer_probs': dict(zip(label_encoder.classes_, probs))
    }

predict_new_film(votes=20000, budget=300_000_000, screens=1500, rating=7.2)

## 11. Conclusion

Ce notebook met en évidence les relations entre les variables quantitatives étudiées et le succès commercial. Il fournit également deux modèles permettant d'estimer le potentiel de succès de nouveaux films à partir de leur profil quantitatif.